<a href="https://colab.research.google.com/github/e23046/Statistical-Learning-e23046/blob/main/Assignment%2003/Q3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Answer For The Question 03

## Spindle Temperature and Power Analysis

This notebook addresses the problem of estimating the internal temperature ($X$) of a CNC spindle based on the observed total spindle power ($Y$), which is affected by mechanical load ($Q$). We'll cover joint distributions, conditional expectations, information geometry, and information theory concepts.

In [ ]:
import numpy as np
from scipy.stats import norm
import math

# Given parameters
mu_X = 0
var_X = 1
std_X = math.sqrt(var_X)

mu_Q = 0
var_Q = 4
std_Q = math.sqrt(var_Q)

# Y = X + Q

### 1. Joint Distribution

**Distribution of $Y$:**
Given $X \sim \mathscr{N}(0, 1)$ and $Q \sim \mathscr{N}(0, 4)$ are independent normal random variables.
Since $Y = X + Q$ and $X, Q$ are independent normal variables, $Y$ will also be normally distributed.

*   Mean of $Y$: $E[Y] = E[X + Q] = E[X] + E[Q] = 0 + 0 = 0$.
*   Variance of $Y$: $\text{Var}(Y) = \text{Var}(X + Q) = \text{Var}(X) + \text{Var}(Q)$ (due to independence) $= 1 + 4 = 5$.

Thus, $Y \sim \mathscr{N}(0, 5)$.

**Covariance $\text{Cov}(X, Y)$:**
$\text{Cov}(X, Y) = \text{Cov}(X, X + Q)$
$= \text{Cov}(X, X) + \text{Cov}(X, Q)$
$= \text{Var}(X) + 0$ (since $X$ and $Q$ are independent, $\text{Cov}(X, Q) = 0$)
$= 1$

So, $\text{Cov}(X, Y) = 1$.

In [ ]:
# Calculate and verify Y properties
mu_Y = mu_X + mu_Q
var_Y = var_X + var_Q # Due to independence

print(f"Mean of Y: {mu_Y}")
print(f"Variance of Y: {var_Y}")

# Calculate and verify Cov(X, Y)
cov_XY = var_X # Since Cov(X, Q) = 0 due to independence
print(f"Cov(X, Y): {cov_XY}")

Mean of Y: 0
Variance of Y: 5
Cov(X, Y): 1


### 2. Information Geometry

**Why $X$ is not $\sigma(Y)$-measurable:**

For $X$ to be $\sigma(Y)$-measurable, it would mean that $X$ can be determined exactly by knowing $Y$. In other words, there would be a deterministic function $f$ such that $X = f(Y)$.

However, we have $Y = X + Q$. Since $Q$ is a random variable (representing noise from mechanical load) with non-zero variance (specifically, $\text{Var}(Q) = 4$), knowing $Y$ does not uniquely determine $X$. For a given value of $Y$, $X$ could take on various values depending on the value of $Q$. There is inherent uncertainty in $X$ even when $Y$ is known.

**Practical terms:** If the power meter shows $Y = 5$, the engineer cannot know for certain if:
*   The motor is overheating ($X$ is high, e.g., $X=4$) and the material is not very hard ($Q$ is small, e.g., $Q=1$).
*   Or if the motor is not overheating ($X$ is normal, e.g., $X=1$) but the material being cut is very hard ($Q$ is high, e.g., $Q=4$).

The randomness contributed by $Q$ makes it impossible to perfectly infer $X$ from $Y$. This is why $X$ is not $\sigma(Y)$-measurable.

### 3. Conditional Expectation (The Best Estimate)

We compute the conditional expectation $E[X \mid Y]$ using the property for joint normals:

$E[X \mid Y = y] = \mu_X + \frac{\text{Cov}(X,Y)}{\text{Var}(Y)}(y - \mu_Y)$

Substituting the values we found:
*   $\mu_X = 0$
*   $\mu_Y = 0$
*   $\text{Cov}(X,Y) = 1$
*   $\text{Var}(Y) = 5$

$E[X \mid Y = y] = 0 + \frac{1}{5}(y - 0) = \frac{1}{5}y$

**Express as a "signal-to-total-variance" ratio:**

The term $\frac{\text{Cov}(X,Y)}{\text{Var}(Y)}$ can be rewritten as:
$\frac{\text{Cov}(X,X+Q)}{\text{Var}(X+Q)} = \frac{\text{Var}(X) + \text{Cov}(X,Q)}{\text{Var}(X) + \text{Var}(Q) + 2\text{Cov}(X,Q)}$
Since $X$ and $Q$ are independent, $\text{Cov}(X,Q) = 0$.
So, $\frac{\text{Var}(X)}{\text{Var}(X) + \text{Var}(Q)} = \frac{1}{1 + 4} = \frac{1}{5}$.

Thus, $E[X \mid Y] = \left(\frac{\text{Var}(X)}{\text{Var}(X) + \text{Var}(Q)}\right) Y$.

In [ ]:
# Calculate the coefficient for E[X | Y]
coefficient = cov_XY / var_Y
print(f"E[X | Y] = {coefficient} * Y")

# Express as signal-to-total-variance ratio
signal_to_total_variance_ratio = var_X / (var_X + var_Q)
print(f"Signal-to-total-variance ratio: {signal_to_total_variance_ratio}")

E[X | Y] = 0.2 * Y
Signal-to-total-variance ratio: 0.2


### 4. Verification of the Residual

Let $\widehat{X} = E[X \mid Y]$. From Question 3, we have $\widehat{X} = \frac{1}{5}Y$.
Define the estimation error (residual) as $Z = X - \widehat{X}$.

**Prove that $E[Z] = 0$:**
$E[Z] = E[X - \widehat{X}] = E[X - \frac{1}{5}Y]$
Using the linearity of expectation:
$E[Z] = E[X] - \frac{1}{5}E[Y]$
Since $E[X] = 0$ and $E[Y] = 0$ (from Question 1):
$E[Z] = 0 - \frac{1}{5}(0) = 0$.

**Verify that the error $Z$ is uncorrelated with the observation $Y$ (Orthogonality Principle):**
We need to show that $\text{Cov}(Z, Y) = 0$.
$\text{Cov}(Z, Y) = \text{Cov}(X - \frac{1}{5}Y, Y)$
Using the properties of covariance:
$\text{Cov}(Z, Y) = \text{Cov}(X, Y) - \text{Cov}(\frac{1}{5}Y, Y)$
$= \text{Cov}(X, Y) - \frac{1}{5}\text{Var}(Y)$
Substituting the values we found:
$= 1 - \frac{1}{5}(5)$
$= 1 - 1 = 0$.

Thus, $Z$ is uncorrelated with $Y$. This demonstrates the Orthogonality Principle, which states that the estimation error is uncorrelated with the data used to make the estimate.

### 5. Information Theory: Differential Entropy

For a normal distribution $N(\mu, \sigma^2)$, the differential entropy is $h(X) = \frac{1}{2}\log_e(2\pi e \sigma^2)$.

**Compute the differential entropy $h(X)$:**
$X \sim \mathscr{N}(0, 1)$, so $\sigma_X^2 = 1$.
$h(X) = \frac{1}{2}\log_e(2\pi e \cdot 1) = \frac{1}{2}\log_e(2\pi e)$.

**Compute the conditional differential entropy $h(X \mid Y)$:**
First, we need the variance of the conditional distribution $(X \mid Y=y)$, which is given by:
$\sigma_{X|Y}^2 = \sigma_X^2 - \frac{\text{Cov}(X,Y)^2}{\text{Var}(Y)}$
Substituting the values:
$\sigma_{X|Y}^2 = 1 - \frac{1^2}{5} = 1 - \frac{1}{5} = \frac{4}{5} = 0.8$

Now, we can compute $h(X \mid Y)$:
$h(X \mid Y) = \frac{1}{2}\log_e(2\pi e \sigma_{X|Y}^2) = \frac{1}{2}\log_e(2\pi e \cdot 0.8)$.

In [ ]:
# Compute h(X)
h_X = 0.5 * np.log(2 * np.pi * np.e * var_X)
print(f"Differential entropy h(X): {h_X:.4f} nats")

# Compute conditional variance sigma_X|Y^2
var_X_given_Y = var_X - (cov_XY**2 / var_Y)
print(f"Conditional variance sigma_X|Y^2: {var_X_given_Y:.4f}")

# Compute h(X | Y)
h_X_given_Y = 0.5 * np.log(2 * np.pi * np.e * var_X_given_Y)
print(f"Conditional differential entropy h(X | Y): {h_X_given_Y:.4f} nats")

Differential entropy h(X): 1.4189 nats
Conditional variance sigma_X|Y^2: 0.8000
Conditional differential entropy h(X | Y): 1.3074 nats


### 6. Mutual Information

**Calculate $I(X; Y) = h(X) - h(X \mid Y)$:**

$I(X; Y) = \frac{1}{2}\log_e(2\pi e) - \frac{1}{2}\log_e(2\pi e \cdot 0.8)$
$I(X; Y) = \frac{1}{2} \left( \log_e(2\pi e) - \log_e(2\pi e \cdot 0.8) \right)$
$I(X; Y) = \frac{1}{2} \log_e\left(\frac{2\pi e}{2\pi e \cdot 0.8}\right)$
$I(X; Y) = \frac{1}{2} \log_e\left(\frac{1}{0.8}\right) = \frac{1}{2} \log_e(1.25)$.

This can also be expressed as $I(X;Y) = \frac{1}{2} \log_e\left(\frac{\text{Var}(X)}{\sigma_{X|Y}^2}\right)$.

**What percentage of the "uncertainty" about the temperature is removed by observing the power meter?**
Percentage removed = $\frac{I(X; Y)}{h(X)} \times 100\%$
Percentage removed = $\frac{\frac{1}{2} \log_e(1.25)}{\frac{1}{2}\log_e(2\pi e)} \times 100\% = \frac{\log_e(1.25)}{\log_e(2\pi e)} \times 100\%$.

**How would this change if the mechanical noise $Q$ had a variance of $100$ instead of $4$?**

If $\text{Var}(Q) = 100$:
*   $\text{Var}(X) = 1$
*   New $\text{Var}(Y) = \text{Var}(X) + \text{Var}(Q) = 1 + 100 = 101$
*   $\text{Cov}(X, Y) = \text{Var}(X) = 1$

New conditional variance $\sigma_{X|Y}^2 = \sigma_X^2 - \frac{\text{Cov}(X,Y)^2}{\text{Var}(Y)} = 1 - \frac{1^2}{101} = 1 - \frac{1}{101} = \frac{100}{101}$.

New Mutual Information $I(X; Y)_{\text{new}} = \frac{1}{2} \log_e\left(\frac{\text{Var}(X)}{\sigma_{X|Y}^2}\right) = \frac{1}{2} \log_e\left(\frac{1}{100/101}\right) = \frac{1}{2} \log_e\left(\frac{101}{100}\right) = \frac{1}{2} \log_e(1.01)$.

Comparing $I(X; Y)_{\text{original}} = \frac{1}{2} \log_e(1.25)$ with $I(X; Y)_{\text{new}} = \frac{1}{2} \log_e(1.01)$, we see that the mutual information *decreases* significantly. This means that if the mechanical noise (Q) is much larger, observing the power meter ($Y$) provides *less* information about the true internal temperature ($X$). The uncertainty about $X$ that is removed by observing $Y$ becomes much smaller, as the signal-to-noise ratio effectively diminishes.

In [ ]:
# Calculate Mutual Information I(X; Y)
I_XY = h_X - h_X_given_Y
print(f"Mutual Information I(X; Y): {I_XY:.4f} nats")

# Percentage of uncertainty removed
percentage_removed = (I_XY / h_X) * 100
print(f"Percentage of uncertainty about X removed by observing Y: {percentage_removed:.2f}%")

# Scenario: Var(Q) = 100
var_Q_new = 100
var_Y_new = var_X + var_Q_new
cov_XY_new = var_X # Still 1 as X and Q are independent

var_X_given_Y_new = var_X - (cov_XY_new**2 / var_Y_new)
h_X_given_Y_new = 0.5 * np.log(2 * np.pi * np.e * var_X_given_Y_new)
I_XY_new = h_X - h_X_given_Y_new

print(f"\nIf Var(Q) = 100:")
print(f"  New Conditional variance sigma_X|Y^2: {var_X_given_Y_new:.4f}")
print(f"  New Mutual Information I(X; Y): {I_XY_new:.4f} nats")

percentage_removed_new = (I_XY_new / h_X) * 100
print(f"  New percentage of uncertainty about X removed: {percentage_removed_new:.2f}%")

Mutual Information I(X; Y): 0.1116 nats
Percentage of uncertainty about X removed by observing Y: 7.86%

If Var(Q) = 100:
  New Conditional variance sigma_X|Y^2: 0.9901
  New Mutual Information I(X; Y): 0.0050 nats
  New percentage of uncertainty about X removed: 0.35%


### 7. Diagnostic Interpretation

**The factory manager's statement: "Since $Y = X + Q$, our best estimate of temperature should just be $Y$ itself."**

This statement is incorrect because it ignores the presence of the random noise $Q$. If we simply use $Y$ as an estimate for $X$, we are essentially saying $\widehat{X} = Y$. However, we found that the best linear unbiased estimator (and for joint normals, the best estimator overall in the mean-square error sense) is the conditional expectation $E[X \mid Y]$. From Question 3, we calculated $E[X \mid Y] = \frac{1}{5}Y$.

**Why $Y$ is a biased estimator of $X$:**
An estimator $\widehat{\theta}$ for a parameter $\theta$ is unbiased if $E[\widehat{\theta}] = \theta$. If we use $Y$ as an estimator for $X$, then $E[Y]$ should equal $X$. But $E[Y] = E[X + Q] = E[X] + E[Q] = 0 + 0 = 0$. While $E[Y]$ equals $E[X]$, this refers to the *average* value over many observations. For a *single* observation $y$, using $y$ directly as an estimate for $x$ would mean that $E[Y \mid X=x]$ should be $x$. But $E[Y \mid X=x] = E[X+Q \mid X=x] = x + E[Q] = x + 0 = x$. So, $Y$ *is* an unbiased estimator in the sense that its expectation is $X$. However, the manager's intuition is flawed because $Y$ includes the noise $Q$, making it an imprecise, high-variance estimate for $X$.

More critically, the conditional expectation $E[X \mid Y]$ is the optimal estimator. The manager's suggested estimator ($Y$) is suboptimal because it doesn't account for the fact that a large $Y$ could be due to a large $Q$ rather than a large $X$.

**Why $E[X \mid Y]$ "shrinks" the measurement toward the mean (zero):**

We found $E[X \mid Y] = \frac{1}{5}Y$. Since $X \sim \mathscr{N}(0,1)$, its mean is $0$. The coefficient $\frac{1}{5}$ is less than 1. This means that the estimated value of $X$ based on $Y$ will always be *smaller in magnitude* than $Y$ itself.

This phenomenon is known as **shrinkage** or **regression to the mean**. Here's why it happens:

1.  **Noise Presence:** The observation $Y$ is a combination of the true signal $X$ and noise $Q$. When $Y$ takes an extreme value (e.g., $Y=5$), it's possible that $X$ is indeed high, but it's *also* very possible that a significant portion of that high value is due to a large (positive) realization of the noise $Q$. The noise term $Q$ has a much larger variance ($\text{Var}(Q) = 4$) compared to the signal $X$ ($\text{Var}(X) = 1$).
2.  **Optimal Estimation:** The conditional expectation $E[X \mid Y]$ attempts to minimize the mean-squared error. To do this, it accounts for the uncertainty introduced by $Q$. If $Y$ is observed to be large, the estimator